# KEAF-Net Demo

End-to-end walkthrough: build the model, run a forward pass on synthetic data, and inspect the AKF filtering + MHSR multi-hop attention. Replace the synthetic example with real extracted features to run on OK-VQA / A-OKVQA (see `docs/DATA.md`).

In [ ]:
import torch
from keafnet.models import KEAFConfig, KEAFNet
from keafnet.data import SyntheticVQADataset, collate

model = KEAFNet(KEAFConfig(pretrained=False)).eval()
batch = collate([SyntheticVQADataset(n=1)[0]])
logits, aux = model(batch)
print('predicted answer index:', logits.argmax(-1).item())
print('AKF kept', int(aux['keep'][0].sum()), 'triplets')
for h, a in enumerate(aux['attn']):
    print(f'hop {h+1} top node:', a[0].argmax().item())

## Knowledge retrieval
Build a tiny ConceptNet-style graph and retrieve triplets for a question.

In [ ]:
from keafnet.retrieval import KGIndex, KnowledgeRetriever, TripletEmbedder
kg = KGIndex()
for s,r,o in [('dog','IsA','animal'),('dog','HasA','tail'),('animal','IsA','organism')]:
    kg.add(s,r,o,1.0)
r = KnowledgeRetriever(kg, TripletEmbedder(dim=384), top_p=10)
strings, emb, mask = r.retrieve(['dog'], 'what animal is shown?')
print(strings)